

---



---


**TASK-1 — Load & Explore Dataset**

---



---



**MOUNT GOOGLE DRIVE**

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

**DEFINE DATASET PATH**

In [ ]:
from pathlib import Path

DATASET_PATH = Path(
    "/content/drive/My Drive/chest-xray-pneumonia-detection/Dataset/chest_xray/chest_xray"
)

print("Dataset Path Loaded Successfully")

In [ ]:
import os
print(os.listdir(DATASET_PATH))

**IMPORT REQUIRED LIBRARIES**

In [ ]:
# File System Handling
import os
from pathlib import Path
from glob import glob

# Image Processing
import cv2
from PIL import Image

# Data Handling
import pandas as pd

# Visualization
import matplotlib.pyplot as plt

# Random Sampling
import random

**VERIFY DATASET STRUCTURE**

In [ ]:
splits = ['train', 'test', 'val']
classes = ['NORMAL', 'PNEUMONIA']

print("="*50)
print("DATASET STRUCTURE VERIFICATION")
print("="*50)
for split in splits:
    split_path = DATASET_PATH / split
    if split_path.exists():
        print(f"\n {split.upper()} folder found")
        for cls in classes:
            class_path = split_path / cls
            if class_path.exists():
                total_images = len(glob(str(class_path / "*")))
                print(f"   |-- {cls:<12}: {total_images} images")
            else:
                print(f"   |-- Missing class folder: {cls}")
    else:
        print(f"\nMissing split folder: {split}")

**COUNT TOTAL IMAGES**

In [ ]:
dataset_summary = []
for split in splits:
    for cls in classes:
        folder_path = DATASET_PATH / split / cls
        image_files = glob(str(folder_path / "*"))
        dataset_summary.append({
            "Dataset Split": split,
            "Class": cls,
            "Total Images": len(image_files)
        })
summary_df = pd.DataFrame(dataset_summary)
print("\nDATASET SUMMARY\n")
display(summary_df)

**VALIDATE DATASET INTEGRITY**

In [ ]:
corrupted_files = []
zero_byte_files = []
print("\nChecking Dataset Integrity...\n")
for split in splits:
    for cls in classes:
        folder_path = DATASET_PATH / split / cls
        image_files = glob(str(folder_path / "*"))
        for image_path in image_files:
            try:
                # Check zero-byte files
                if os.path.getsize(image_path) == 0:
                    zero_byte_files.append(image_path)
                    continue
                # Verify image validity
                with Image.open(image_path) as img:
                    img.verify()
            except Exception:
                corrupted_files.append(image_path)
print("="*50)
print("DATASET VALIDATION REPORT")
print("="*50)
print(f"Corrupted Images : {len(corrupted_files)}")
print(f"Zero-byte Images : {len(zero_byte_files)}")

**DISPLAY CORRUPTED FILES**

In [ ]:
if corrupted_files:
    print("\nSample Corrupted Files:\n")
    for file in corrupted_files[:5]:
        print(file)
else:
    print("No corrupted files found.")

**Load Random Sample Images**

In [ ]:
sample_images = []
# Number of images per class
images_per_class = 3
for cls in classes:
    folder_path = DATASET_PATH / "train" / cls
    image_files = glob(str(folder_path / "*"))
    # Select random images
    random_samples = random.sample(
        image_files,
        images_per_class
    )
    for image_path in random_samples:
        sample_images.append((cls, image_path))
print(f"Total Sample Images Loaded: {len(sample_images)}")

**Extract Image Metadata**

In [ ]:
metadata = []

for label, image_path in sample_images:
    # Read image using PIL
    pil_img = Image.open(image_path)
    # Read image using OpenCV
    cv_img = cv2.imread(image_path)
    width, height = pil_img.size
    channels = cv_img.shape[2] if len(cv_img.shape) == 3 else 1
    metadata.append({
        "Class": label,
        "Filename": os.path.basename(image_path),
        "Width": width,
        "Height": height,
        "Channels": channels,
        "Format": pil_img.format
    })

metadata_df = pd.DataFrame(metadata)
print("\nIMAGE METADATA\n")
display(metadata_df)

**Display Sample Images**

In [ ]:
plt.figure(figsize=(10,5))
for i, (label, image_path) in enumerate(sample_images):
    img = Image.open(image_path)
    plt.subplot(2, 3, i+1)
    plt.imshow(img, cmap='gray')
    plt.title(label)
    plt.axis("off")
plt.tight_layout()
plt.show()

**Save Reports**

In [ ]:
# Create reports folder
os.makedirs("reports", exist_ok=True)
# Save CSV files
summary_df.to_csv("reports/dataset_summary.csv", index=False)
metadata_df.to_csv("reports/image_metadata.csv", index=False)
print("Reports saved successfully.")

In [ ]:
summary_df.to_json(
    "dataset_summary.json",
    orient="records"
)

In [ ]:
metadata_df.to_json(
    "image_metadata.json",
    orient="records"
)